# 05 — Verification of the numbers of Section 4 (Results)

This notebook **trains nothing**: it only reads the result files saved by `03_nas.ipynb` and recomputes, independently of the other notebooks, every number quoted in Section 4 of the report. Each section of the notebook corresponds to a subsection of the report.

**Inputs:** `risultati_nas_v2.csv` (search, seed 0), `rumore_semi.csv` (seed study, seeds 0–4), `multi_seed.csv` (re-evaluation, seeds 1–5), `risultati_test.csv` and `punteggi_test.npz` (test, seeds 1–5).

**Naming.** `larghezze` = widths of the hidden layers (the name of an architecture), `profondita` = depth, `primo_layer` = width of the first layer, `n_param` = number of parameters, `epoca_migliore` = best epoch, `calo` = drop, `frontiera` = Pareto front, `fascia` = cost band, `confronto` = comparison, `cav` = straddling windows.

To run it, place it in the project folder and execute all cells (*Run → Run All Cells*).

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, roc_auc_score
pd.set_option("display.width", 200)

def se(x):
    """Standard error of the mean: standard deviation (divided by n-1) / square root of n."""
    x = np.asarray(x, dtype=float)
    return x.std(ddof=1) / np.sqrt(len(x))

def confronto(df, a, b, col):
    """Compare the means of two architectures: difference, its standard error, and their ratio."""
    xa = df.loc[df["larghezze"] == a, col].values
    xb = df.loc[df["larghezze"] == b, col].values
    d = xa.mean() - xb.mean()
    se_d = np.sqrt(se(xa)**2 + se(xb)**2)     # the two means are treated as independent
    print(f"{col}: {a} ({xa.mean():.4f} ± {se(xa):.4f}, n={len(xa)})  vs  "
          f"{b} ({xb.mean():.4f} ± {se(xb):.4f}, n={len(xb)})  ->  "
          f"Δ = {d:+.4f}, se_Δ = {se_d:.4f}, Δ/se_Δ = {d/se_d:+.1f}")

def frontiera(stat, col):
    """Pareto set: sort by number of parameters and keep each architecture that beats all the cheaper ones."""
    migliore, fr = -np.inf, []
    for _, r in stat.sort_values("n_param").iterrows():
        if r[col] > migliore:
            fr.append(r["larghezze"])
            migliore = r[col]
    return fr

In [2]:
# The column of the architectures is read as text, so that "16" stays a string
ris = pd.read_csv("risultati_nas_v2.csv", dtype={"larghezze": str})   # search, seed 0
rum = pd.read_csv("rumore_semi.csv", dtype={"larghezze": str})        # seed study, seeds 0-4
ms  = pd.read_csv("multi_seed.csv", dtype={"larghezze": str})         # re-evaluation
te  = pd.read_csv("risultati_test.csv", dtype={"larghezze": str})     # test, seeds 1-5
P   = np.load("punteggi_test.npz")
y_test = P["y_test"]

# Re-evaluation: seeds 1-5 only. Depth and first-layer width are derived from the name.
ms = ms[ms["seed"].isin([1, 2, 3, 4, 5])]
for df in (ris, rum, ms, te):
    df["profondita"] = df["larghezze"].str.count("-") + 1
    df["primo_layer"] = df["larghezze"].str.split("-").str[0].astype(int)

print(len(ris), "architetture nella ricerca |", ms["larghezze"].nunique(), "candidate |",
      te["larghezze"].nunique(), "reti nel test")

300 architetture nella ricerca | 30 candidate | 9 reti nel test


## 4.1 Search phase results

In [3]:
# Validation AUC and AP of the 300 architectures (seed 0)
print("Statistiche delle 300 architetture (seme 0):")
print(ris[["auc_val", "ap_val"]].describe().round(4))

# By depth (Table 7)
print("\nPer profondità:")
print(ris.groupby("profondita")["auc_val"].agg(["count", "mean", "min", "max"]).round(4))

# The ten best architectures, and how many of them have a first layer of 16 units
top10 = ris.sort_values("auc_val", ascending=False).head(10)
print("\nLe 10 migliori:")
print(top10[["larghezze", "n_param", "auc_val"]].to_string(index=False))
print("Con primo layer da 16:", (top10["primo_layer"] == 16).sum())

# Median of the search and position of the baseline
print("\nMediana AUC:", round(ris["auc_val"].median(), 4),
      "| baseline (seme 0):", round(ris.loc[ris["larghezze"] == "64-64", "auc_val"].iloc[0], 4))

Statistiche delle 300 architetture (seme 0):
        auc_val    ap_val
count  300.0000  300.0000
mean     0.8651    0.3617
std      0.0041    0.0290
min      0.8532    0.2856
25%      0.8621    0.3380
50%      0.8648    0.3648
75%      0.8677    0.3869
max      0.8795    0.4130

Per profondità:
            count    mean     min     max
profondita                               
1               5  0.8568  0.8532  0.8593
2              25  0.8623  0.8588  0.8658
3             102  0.8645  0.8566  0.8795
4             168  0.8662  0.8585  0.8761

Le 10 migliori:
    larghezze  n_param  auc_val
     16-16-64     3361 0.879461
 32-16-128-64    14897 0.876077
    16-256-32    14545 0.875672
16-128-16-128     8481 0.875377
 16-32-128-32    10865 0.875215
  16-64-64-16     8241 0.874991
16-64-256-128    52689 0.874726
 16-32-32-128     7889 0.873622
 64-32-32-128    15233 0.873249
 32-32-32-128    10337 0.872926
Con primo layer da 16: 7

Mediana AUC: 0.8648 | baseline (seme 0): 0.8632


In [4]:
# Mean best epoch (seeds 1-5) of small and large candidates (quoted in Sec. 4.3)
ep = ms.groupby("larghezze").agg(n_param=("n_param", "first"), epoca=("epoca_migliore", "mean"))
print("Meno di 6.000 parametri: ", ep[ep["n_param"] < 6000]["epoca"].round(1).sort_values().values)
print("Più di 50.000 parametri:", ep[ep["n_param"] > 50000]["epoca"].round(1).sort_values().values)

Meno di 6.000 parametri:  [11.  16.4 16.4 19.2 20.4 24.4]
Più di 50.000 parametri: [3.2 3.4 3.6 4.2 4.6 4.6 5.4 8.  8.4 8.8]


## 4.2 Seed variance and the winner's curse

In [5]:
# Mean, standard deviation and range of the validation AUC over seeds 0-4 (Table 10)
g = rum.groupby("larghezze")["auc_val"]
tab = pd.DataFrame({"n_param": rum.groupby("larghezze")["n_param"].first(),
                    "media": g.mean(), "dev_std": g.std(), "min": g.min(), "max": g.max()})
print(tab.sort_values("n_param").round(4))
# For comparison: spread between the 300 different architectures of the search
print("\nDeviazione standard tra le 300 architetture:", round(ris["auc_val"].std(), 4))

# The noise does not blur large differences: the worst seed of 16-16-16 and of 16-16-64
# is still above the best seed of the baseline
print("\nMassimo della baseline sui 5 semi:",
      round(rum.loc[rum["larghezze"] == "64-64", "auc_val"].max(), 4))
for k in ["16-16-16", "16-16-64"]:
    print(k, "- minimo sui 5 semi:", round(rum.loc[rum["larghezze"] == k, "auc_val"].min(), 4))
    confronto(rum, k, "64-64", "auc_val")

                 n_param   media  dev_std     min     max
larghezze                                                
16                  1953  0.8584   0.0014  0.8569  0.8602
16-16-16            2497  0.8701   0.0032  0.8659  0.8745
16-16-64            3361  0.8712   0.0053  0.8654  0.8795
32-32-32            6017  0.8648   0.0044  0.8601  0.8705
64-64              11969  0.8631   0.0010  0.8615  0.8642
256-256-256-256   228609  0.8635   0.0040  0.8600  0.8702

Deviazione standard tra le 300 architetture: 0.0041

Massimo della baseline sui 5 semi: 0.8642
16-16-16 - minimo sui 5 semi: 0.8659
auc_val: 16-16-16 (0.8701 ± 0.0014, n=5)  vs  64-64 (0.8631 ± 0.0004, n=5)  ->  Δ = +0.0070, se_Δ = 0.0015, Δ/se_Δ = +4.7
16-16-64 - minimo sui 5 semi: 0.8654
auc_val: 16-16-64 (0.8712 ± 0.0024, n=5)  vs  64-64 (0.8631 ± 0.0004, n=5)  ->  Δ = +0.0081, se_Δ = 0.0024, Δ/se_Δ = +3.4


## 4.3 Pareto front on multi-seed means

In [6]:
# Mean and standard error over seeds 1-5 for the 30 candidates (Table 8, Table 11)
stat = (ms.groupby("larghezze")
          .agg(n_param=("n_param", "first"), n_semi=("seed", "count"),
               auc=("auc_val", "mean"), auc_se=("auc_val", se),
               ap=("ap_val", "mean"), ap_se=("ap_val", se))
          .reset_index().sort_values("n_param"))
# Single-seed AUC of the search and drop with respect to it (positive = worse with new seeds)
stat["auc_seme0"] = stat["larghezze"].map(dict(zip(ris["larghezze"], ris["auc_val"])))
stat["calo"] = stat["auc_seme0"] - stat["auc"]
stat["profondita"] = stat["larghezze"].str.count("-") + 1
stat["primo_layer"] = stat["larghezze"].str.split("-").str[0].astype(int)

# Pareto fronts on the mean AUC and on the mean AP
fr_auc = frontiera(stat, "auc")
fr_ap = frontiera(stat, "ap")
stat["front_auc"] = stat["larghezze"].isin(fr_auc)
stat["front_ap"] = stat["larghezze"].isin(fr_ap)
print(stat.drop(columns=["profondita", "primo_layer"]).to_string(index=False, float_format="%.4f"))

      larghezze  n_param  n_semi    auc  auc_se     ap  ap_se  auc_seme0    calo  front_auc  front_ap
             16     1953       5 0.8584  0.0006 0.3107 0.0052     0.8593  0.0009       True      True
       16-16-16     2497       5 0.8706  0.0014 0.3849 0.0114     0.8688 -0.0018       True      True
    16-32-16-16     3297       5 0.8714  0.0020 0.3951 0.0086     0.8694 -0.0020       True      True
       16-16-64     3361       5 0.8688  0.0012 0.3920 0.0024     0.8795  0.0107      False     False
       16-32-32     3569       5 0.8665  0.0016 0.3711 0.0090     0.8714  0.0049      False     False
    16-32-32-64     5713       5 0.8698  0.0017 0.4008 0.0033     0.8707  0.0009      False      True
       32-32-32     6017       5 0.8646  0.0018 0.3544 0.0101     0.8705  0.0059      False     False
      32-16-128     6705       5 0.8697  0.0012 0.3889 0.0046     0.8716  0.0019      False     False
   16-32-32-128     7889       5 0.8699  0.0012 0.3927 0.0127     0.8736  0.0037  

In [7]:
# Steps along the AUC front: each architecture against the previous one on the front
print("Frontiera AUC:", fr_auc)
for a, b in zip(fr_auc[1:], fr_auc[:-1]):
    confronto(ms, a, b, "auc_val")

# Steps along the AP front
print("\nFrontiera AP:", fr_ap)
for a, b in zip(fr_ap[1:], fr_ap[:-1]):
    confronto(ms, a, b, "ap_val")

# Comparisons quoted in the text
print("\nConfronti citati nel testo:")
confronto(ms, "16-16-16", "64-64", "auc_val")
confronto(ms, "16-128-16-128", "64-64", "auc_val")
confronto(ms, "16-128-16-128", "64-64", "ap_val")
confronto(ms, "256-256-256-256", "64-64", "auc_val")
confronto(ms, "16-128-16-128", "16-16-16", "ap_val")

Frontiera AUC: ['16', '16-16-16', '16-32-16-16', '16-128-16-128', '16-64-256-128']
auc_val: 16-16-16 (0.8706 ± 0.0014, n=5)  vs  16 (0.8584 ± 0.0006, n=5)  ->  Δ = +0.0121, se_Δ = 0.0015, Δ/se_Δ = +7.8
auc_val: 16-32-16-16 (0.8714 ± 0.0020, n=5)  vs  16-16-16 (0.8706 ± 0.0014, n=5)  ->  Δ = +0.0008, se_Δ = 0.0024, Δ/se_Δ = +0.3
auc_val: 16-128-16-128 (0.8718 ± 0.0017, n=5)  vs  16-32-16-16 (0.8714 ± 0.0020, n=5)  ->  Δ = +0.0004, se_Δ = 0.0026, Δ/se_Δ = +0.2
auc_val: 16-64-256-128 (0.8731 ± 0.0021, n=5)  vs  16-128-16-128 (0.8718 ± 0.0017, n=5)  ->  Δ = +0.0013, se_Δ = 0.0027, Δ/se_Δ = +0.5

Frontiera AP: ['16', '16-16-16', '16-32-16-16', '16-32-32-64', '16-128-16-128', '16-256-256-64']
ap_val: 16-16-16 (0.3849 ± 0.0114, n=5)  vs  16 (0.3107 ± 0.0052, n=5)  ->  Δ = +0.0742, se_Δ = 0.0125, Δ/se_Δ = +6.0
ap_val: 16-32-16-16 (0.3951 ± 0.0086, n=5)  vs  16-16-16 (0.3849 ± 0.0114, n=5)  ->  Δ = +0.0102, se_Δ = 0.0142, Δ/se_Δ = +0.7
ap_val: 16-32-32-64 (0.4008 ± 0.0033, n=5)  vs  16-32-16-16

In [8]:
# Winner's curse: drop of the 28 candidates selected for their single-seed AUC
# (16 and the baseline were added for other reasons and are excluded)
sel = stat[~stat["larghezze"].isin(["16", "64-64"])]
print("Candidate selezionate:", len(sel), "| che calano:", (sel["calo"] > 0).sum(),
      "| calo medio:", round(sel["calo"].mean(), 4))
print(stat[stat["larghezze"].isin(["16", "64-64", "16-16-64"])][["larghezze", "calo"]]
      .round(4).to_string(index=False))

# Rank of 16-16-16 and 16-32-16-16 in their cost band (seed 0)
BORDI = [1900, 4000, 8000, 16000, 32000, 64000, 128000, 256000]
ris["fascia"] = pd.cut(ris["n_param"], bins=BORDI)
f1 = ris.loc[ris["larghezze"] == "16-16-16", "fascia"].iloc[0]
print("\nPrima fascia, ordinata per AUC del seme 0:")
print(ris[ris["fascia"] == f1].sort_values("auc_val", ascending=False)
      .head(5)[["larghezze", "n_param", "auc_val"]].to_string(index=False))

# Candidates with fewer parameters than the baseline
base = stat.loc[stat["larghezze"] == "64-64"].iloc[0]
piccole = stat[(stat["n_param"] < base["n_param"]) & (stat["larghezze"] != "16")]
print("\nCandidate più piccole della baseline (esclusa 16):", len(piccole),
      "| con AUC media più alta:", (piccole["auc"] > base["auc"]).sum())

# Width of the first layer (only networks with more than one layer)
multi = stat[stat["profondita"] > 1]
for h in [16, 256]:
    sub = multi[multi["primo_layer"] == h]["auc"]
    print(f"Primo layer {h}: AUC media da {sub.min():.4f} a {sub.max():.4f}")

Candidate selezionate: 28 | che calano: 24 | calo medio: 0.0039
larghezze   calo
       16 0.0009
 16-16-64 0.0107
    64-64 0.0001

Prima fascia, ordinata per AUC del seme 0:
  larghezze  n_param  auc_val
   16-16-64     3361 0.879461
   16-32-32     3569 0.871435
16-32-16-16     3297 0.869393
   16-16-16     2497 0.868828
   16-32-16     3025 0.866949

Candidate più piccole della baseline (esclusa 16): 10 | con AUC media più alta: 10
Primo layer 16: AUC media da 0.8665 a 0.8731
Primo layer 256: AUC media da 0.8611 a 0.8619


## 4.4 Test set evaluation

In [9]:
# Test AUC and AP: mean and standard error over seeds 1-5 (Table 9)
tt = (te.groupby("larghezze")
        .agg(n_param=("n_param", "first"),
             auc=("auc_test", "mean"), auc_se=("auc_test", se),
             ap=("ap_test", "mean"), ap_se=("ap_test", se))
        .reset_index().sort_values("n_param"))
print(tt.to_string(index=False, float_format="%.4f"))

# Reproducibility: validation AUC of the test runs against that of the re-evaluation
chk = te.merge(ms[["larghezze", "seed", "auc_val"]], on=["larghezze", "seed"],
               suffixes=("_test", "_rival"))
print("\nRun confrontati:", len(chk),
      "| differenza massima di AUC di validation:",
      (chk["auc_val_test"] - chk["auc_val_rival"]).abs().max())

# Range of the AUC on the test set and, for the same nine architectures, on the validation set
print("Intervallo AUC test:", round(tt["auc"].min(), 4), "-", round(tt["auc"].max(), 4))
v9 = stat[stat["larghezze"].isin(tt["larghezze"])]
print("Intervallo AUC validation (stesse 9):", round(v9["auc"].min(), 4), "-", round(v9["auc"].max(), 4))

      larghezze  n_param    auc  auc_se     ap  ap_se
             16     1953 0.8861  0.0013 0.2924 0.0055
       16-16-16     2497 0.8972  0.0044 0.4166 0.0176
    16-32-16-16     3297 0.9025  0.0025 0.4310 0.0065
    16-32-32-64     5713 0.9009  0.0040 0.4423 0.0062
  16-128-16-128     8481 0.9022  0.0016 0.4572 0.0036
          64-64    11969 0.8918  0.0008 0.3281 0.0040
  16-64-256-128    52689 0.9035  0.0025 0.4413 0.0148
  16-256-256-64    88593 0.9025  0.0010 0.4621 0.0083
256-256-256-256   228609 0.8916  0.0015 0.3477 0.0124

Run confrontati: 45 | differenza massima di AUC di validation: 0.0
Intervallo AUC test: 0.8861 - 0.9035
Intervallo AUC validation (stesse 9): 0.8584 - 0.8731


In [10]:
# Comparisons with the baseline on the test set
for k in ["16-128-16-128", "16-32-16-16", "16-16-16", "256-256-256-256"]:
    confronto(te, k, "64-64", "auc_test")
    confronto(te, k, "64-64", "ap_test")
    print()
# Position of the knee of the AP
confronto(te, "16-128-16-128", "16-32-16-16", "ap_test")
confronto(te, "16-256-256-64", "16-128-16-128", "ap_test")

# The five seeds of 16-16-16, the least stable network
print("\nSemi di 16-16-16:")
print(te[te["larghezze"] == "16-16-16"][["seed", "auc_test", "ap_test"]].round(4).to_string(index=False))

auc_test: 16-128-16-128 (0.9022 ± 0.0016, n=5)  vs  64-64 (0.8918 ± 0.0008, n=5)  ->  Δ = +0.0104, se_Δ = 0.0018, Δ/se_Δ = +5.9
ap_test: 16-128-16-128 (0.4572 ± 0.0036, n=5)  vs  64-64 (0.3281 ± 0.0040, n=5)  ->  Δ = +0.1291, se_Δ = 0.0054, Δ/se_Δ = +24.1

auc_test: 16-32-16-16 (0.9025 ± 0.0025, n=5)  vs  64-64 (0.8918 ± 0.0008, n=5)  ->  Δ = +0.0107, se_Δ = 0.0026, Δ/se_Δ = +4.0
ap_test: 16-32-16-16 (0.4310 ± 0.0065, n=5)  vs  64-64 (0.3281 ± 0.0040, n=5)  ->  Δ = +0.1029, se_Δ = 0.0077, Δ/se_Δ = +13.4

auc_test: 16-16-16 (0.8972 ± 0.0044, n=5)  vs  64-64 (0.8918 ± 0.0008, n=5)  ->  Δ = +0.0054, se_Δ = 0.0045, Δ/se_Δ = +1.2
ap_test: 16-16-16 (0.4166 ± 0.0176, n=5)  vs  64-64 (0.3281 ± 0.0040, n=5)  ->  Δ = +0.0885, se_Δ = 0.0180, Δ/se_Δ = +4.9

auc_test: 256-256-256-256 (0.8916 ± 0.0015, n=5)  vs  64-64 (0.8918 ± 0.0008, n=5)  ->  Δ = -0.0002, se_Δ = 0.0016, Δ/se_Δ = -0.1
ap_test: 256-256-256-256 (0.3477 ± 0.0124, n=5)  vs  64-64 (0.3281 ± 0.0040, n=5)  ->  Δ = +0.0196, se_Δ = 0.0130,

In [11]:
# Precision at a recall of 50%, and false-positive windows for every positive window detected
def prec_a_recall(y, p, r=0.5):
    """Highest precision among the thresholds with a recall of at least r."""
    prec, rec, _ = precision_recall_curve(y, p)
    return prec[rec >= r].max()

righe = []
for k in tt["larghezze"]:
    v = [prec_a_recall(y_test, P[f"{k}_seed{s}_test"]) for s in [1, 2, 3, 4, 5]]
    righe.append({"larghezze": k, "prec_rec50": np.mean(v), "se": se(v)})
pr = pd.DataFrame(righe)
# With precision p, there are (1 - p) / p false positives for every true positive
pr["fp_per_tp"] = (1 - pr["prec_rec50"]) / pr["prec_rec50"]
print(pr.to_string(index=False, float_format="%.3f"))

fp_base = pr.loc[pr["larghezze"] == "64-64", "fp_per_tp"].iloc[0]
fp_best = pr.loc[pr["larghezze"] == "16-128-16-128", "fp_per_tp"].iloc[0]
print("\nRiduzione dei falsi positivi di 16-128-16-128 rispetto alla baseline:",
      f"{100 * (1 - fp_best / fp_base):.0f}%")

      larghezze  prec_rec50    se  fp_per_tp
             16       0.287 0.008      2.484
       16-16-16       0.386 0.014      1.589
    16-32-16-16       0.409 0.017      1.446
    16-32-32-64       0.414 0.008      1.417
  16-128-16-128       0.437 0.011      1.291
          64-64       0.327 0.005      2.060
  16-64-256-128       0.441 0.015      1.270
  16-256-256-64       0.459 0.021      1.179
256-256-256-256       0.314 0.015      2.182

Riduzione dei falsi positivi di 16-128-16-128 rispetto alla baseline: 37%


## 4.5 Control on straddling windows

In [12]:
neg, pos = (y_test == 0), (y_test == 1)
righe = []
for k in tt["larghezze"]:
    for s in [1, 2, 3, 4, 5]:
        # Scores of this run on the test windows and on the straddling windows
        p_te, p_cav = P[f"{k}_seed{s}_test"], P[f"{k}_seed{s}_cav"]
        # Straddling windows (label 1) against the negative test windows (label 0)
        y_mix = np.r_[np.ones(len(p_cav)), np.zeros(neg.sum())]
        righe.append({"larghezze": k,
                      "auc_cav": roc_auc_score(y_mix, np.r_[p_cav, p_te[neg]]),
                      "med_neg": np.median(p_te[neg]), "med_pos": np.median(p_te[pos]),
                      "med_cav": np.median(p_cav)})
# Mean over the five seeds, and range of each quantity across the nine architectures
cav = pd.DataFrame(righe).groupby("larghezze").mean()
print("Finestre a cavallo:", len(P["64-64_seed1_cav"]))
print(cav.round(3))
for c in cav.columns:
    print(f"{c}: da {cav[c].min():.3f} a {cav[c].max():.3f}")

Finestre a cavallo: 468
                 auc_cav  med_neg  med_pos  med_cav
larghezze                                          
16                 0.984    0.148    0.875    0.982
16-128-16-128      0.951    0.074    0.935    0.983
16-16-16           0.967    0.111    0.920    0.976
16-256-256-64      0.969    0.112    0.938    0.979
16-32-16-16        0.954    0.081    0.924    0.981
16-32-32-64        0.942    0.103    0.933    0.977
16-64-256-128      0.965    0.100    0.932    0.980
256-256-256-256    0.971    0.123    0.869    0.958
64-64              0.980    0.139    0.881    0.977
auc_cav: da 0.942 a 0.984
med_neg: da 0.074 a 0.148
med_pos: da 0.869 a 0.938
med_cav: da 0.958 a 0.983
